# Skin Cancer Classification — Training Notebook (Google Colab)

**DDM501 Final Project** — End-to-end ML system for dermoscopic image classification.

This notebook trains 4 models on the ISIC 9-class skin cancer dataset using a T4/A100 GPU:
- Custom CNN
- ResNet50 (transfer learning)
- EfficientNet-B0 (transfer learning)
- ViT-B/16 (Vision Transformer)

**Instructions:**
1. Set runtime to **GPU** (Runtime → Change runtime type → T4 GPU)
2. Run all cells in order
3. Download the trained `.pth` files from the `models/` folder when done

## 1. Setup & Install Dependencies

In [ ]:
!pip install -q torch torchvision timm Augmentor kagglehub scikit-learn

In [ ]:
import os
import logging
import time
import shutil
import tempfile
from pathlib import Path
from typing import Tuple, Dict, Optional

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms, models
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import timm

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger(__name__)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Download Dataset from Kaggle

In [ ]:
import kagglehub

dataset_path = kagglehub.dataset_download("nodoubttome/skin-cancer9-classesisic")
print(f"Dataset downloaded to: {dataset_path}")

# Show contents
for item in sorted(Path(dataset_path).rglob("*")):
    if item.is_dir() and item.parent == Path(dataset_path):
        print(f"  {item.name}/")

## 3. Data Pipeline

In [ ]:
# ===================== Constants =====================
CLASS_NAMES = [
    "actinic keratosis",
    "basal cell carcinoma",
    "dermatofibroma",
    "melanoma",
    "nevus",
    "pigmented benign keratosis",
    "seborrheic keratosis",
    "squamous cell carcinoma",
    "vascular lesion",
]
MALIGNANT_CLASSES = {"melanoma", "basal cell carcinoma", "squamous cell carcinoma"}
NUM_CLASSES = len(CLASS_NAMES)
IMG_SIZE = 224

print(f"Number of classes: {NUM_CLASSES}")
print(f"Classes: {CLASS_NAMES}")

In [ ]:
# ===================== Dataset Class =====================
class ISICSkinDataset(Dataset):
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        label = self.labels[idx]
        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, label


# ===================== Transforms =====================
def get_transforms(split="train"):
    if split == "train":
        return transforms.Compose([
            transforms.Resize((IMG_SIZE + 32, IMG_SIZE + 32)),
            transforms.RandomCrop(IMG_SIZE),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomVerticalFlip(p=0.5),
            transforms.RandomRotation(degrees=30),
            transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
            transforms.RandomAffine(degrees=15, translate=(0.1, 0.1), scale=(0.9, 1.1)),
            transforms.RandomGrayscale(p=0.05),
            transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
            transforms.RandomErasing(p=0.2),
        ])
    else:
        return transforms.Compose([
            transforms.Resize((IMG_SIZE, IMG_SIZE)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])

In [ ]:
# ===================== Data Loading =====================
def _resolve_data_path(data_dir):
    data_path = Path(data_dir)
    nested = data_path / "Skin cancer ISIC The International Skin Imaging Collaboration"
    if nested.exists():
        data_path = nested
    return data_path


def _find_split_dirs(data_path):
    train_dir = test_dir = None
    for child in data_path.iterdir():
        if child.is_dir() and child.name.lower() == "train":
            train_dir = child
        elif child.is_dir() and child.name.lower() == "test":
            test_dir = child
    if train_dir is None and test_dir is None:
        raise FileNotFoundError(f"No Train/ or Test/ directory found in {data_path}")
    return train_dir, test_dir


def _load_folder(folder):
    class_to_idx = {name.lower(): idx for idx, name in enumerate(CLASS_NAMES)}
    image_paths, labels = [], []
    for class_folder in sorted(folder.iterdir()):
        if not class_folder.is_dir():
            continue
        class_name = class_folder.name.lower().strip()
        if class_name not in class_to_idx:
            logger.warning(f"Unknown class folder '{class_folder.name}', skipping")
            continue
        label = class_to_idx[class_name]
        for img_file in class_folder.iterdir():
            if img_file.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp"}:
                image_paths.append(str(img_file))
                labels.append(label)
    return image_paths, labels


def load_train_test_split(data_dir):
    data_path = _resolve_data_path(data_dir)
    train_dir, test_dir = _find_split_dirs(data_path)
    result = {}
    if train_dir:
        result["train_all"] = _load_folder(train_dir)
        logger.info(f"Train folder: {len(result['train_all'][0])} images")
    if test_dir:
        result["test"] = _load_folder(test_dir)
        logger.info(f"Test folder: {len(result['test'][0])} images")
    return result

In [ ]:
# ===================== Class Balancing (Augmentor) =====================
import Augmentor

def _balance_classes(image_paths, labels, samples_per_class=1000):
    class_to_paths = {}
    for path, label in zip(image_paths, labels):
        class_to_paths.setdefault(label, []).append(path)

    balanced_paths, balanced_labels = [], []

    # Use a persistent directory so augmented images are reused across runs
    augmented_dir = Path("augmented_data")
    augmented_dir.mkdir(parents=True, exist_ok=True)

    for label_idx in range(NUM_CLASSES):
        class_name = CLASS_NAMES[label_idx]
        paths = class_to_paths.get(label_idx, [])
        count = len(paths)

        balanced_paths.extend(paths)
        balanced_labels.extend([label_idx] * count)

        if count >= samples_per_class:
            logger.info(f"Class '{class_name}': {count} images (no augmentation needed)")
            continue

        needed = samples_per_class - count
        class_output_dir = augmented_dir / f"output_{label_idx}"

        # Check if augmented images already exist
        existing = sorted(class_output_dir.glob("*.*")) if class_output_dir.exists() else []
        if len(existing) >= needed:
            logger.info(f"Class '{class_name}': {count} images, reusing {needed} existing augmented images")
            for gen_path in existing[:needed]:
                balanced_paths.append(str(gen_path))
                balanced_labels.append(label_idx)
            continue

        logger.info(f"Class '{class_name}': {count} images, generating {needed} augmented images")

        class_input_dir = augmented_dir / f"input_{label_idx}"
        class_input_dir.mkdir(parents=True, exist_ok=True)
        class_output_dir.mkdir(parents=True, exist_ok=True)

        for i, src_path in enumerate(paths):
            ext = Path(src_path).suffix
            dst = class_input_dir / f"img_{i}{ext}"
            if not dst.exists():
                shutil.copy2(src_path, dst)

        p = Augmentor.Pipeline(source_directory=str(class_input_dir), output_directory=str(class_output_dir))
        p.rotate(probability=0.7, max_left_rotation=15, max_right_rotation=15)
        p.flip_left_right(probability=0.5)
        p.flip_top_bottom(probability=0.5)
        p.zoom_random(probability=0.5, percentage_area=0.8)
        p.random_distortion(probability=0.3, grid_width=4, grid_height=4, magnitude=2)
        p.sample(needed)

        generated = sorted(class_output_dir.glob("*.*"))
        for gen_path in generated[:needed]:
            balanced_paths.append(str(gen_path))
            balanced_labels.append(label_idx)

    final_counts = np.bincount(balanced_labels, minlength=NUM_CLASSES)
    logger.info(f"Balanced dataset: {len(balanced_paths)} total images")
    for i, name in enumerate(CLASS_NAMES):
        logger.info(f"  {name}: {final_counts[i]}")

    return balanced_paths, balanced_labels

In [ ]:
# ===================== Utility Functions =====================
def compute_class_weights(labels):
    class_counts = np.bincount(labels, minlength=NUM_CLASSES)
    total = len(labels)
    weights = total / (NUM_CLASSES * class_counts.astype(float) + 1e-6)
    return torch.FloatTensor(weights)


def get_weighted_sampler(labels):
    class_counts = np.bincount(labels, minlength=NUM_CLASSES)
    class_weights = 1.0 / (class_counts.astype(float) + 1e-6)
    sample_weights = [class_weights[label] for label in labels]
    return WeightedRandomSampler(weights=sample_weights, num_samples=len(labels), replacement=True)


def split_dataset(image_paths, labels, val_size=0.15, random_state=42):
    train_paths, val_paths, train_labels, val_labels = train_test_split(
        image_paths, labels, test_size=val_size, stratify=labels, random_state=random_state
    )
    logger.info(f"Split sizes - Train: {len(train_paths)}, Val: {len(val_paths)}")
    return {"train": (train_paths, train_labels), "val": (val_paths, val_labels)}

In [ ]:
# ===================== Create DataLoaders =====================
def create_data_loaders(data_dir, batch_size=32, num_workers=2, balance_classes=True, samples_per_class=1000):
    data = load_train_test_split(data_dir)
    train_all_paths, train_all_labels = data["train_all"]
    splits = split_dataset(train_all_paths, train_all_labels)

    train_paths, train_labels = splits["train"]
    val_paths, val_labels = splits["val"]
    test_paths, test_labels = data["test"]

    if balance_classes:
        train_paths, train_labels = _balance_classes(train_paths, train_labels, samples_per_class)

    class_weights = compute_class_weights(train_labels)

    train_dataset = ISICSkinDataset(train_paths, train_labels, get_transforms("train"))
    val_dataset = ISICSkinDataset(val_paths, val_labels, get_transforms("val"))
    test_dataset = ISICSkinDataset(test_paths, test_labels, get_transforms("test"))

    train_sampler = get_weighted_sampler(train_labels)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, sampler=train_sampler,
                              num_workers=num_workers, pin_memory=True, drop_last=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False,
                            num_workers=num_workers, pin_memory=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False,
                             num_workers=num_workers, pin_memory=True)

    logger.info(f"DataLoaders - Train batches: {len(train_loader)}, Val: {len(val_loader)}, Test: {len(test_loader)}")
    return train_loader, val_loader, test_loader, class_weights


# Create loaders now
train_loader, val_loader, test_loader, class_weights = create_data_loaders(dataset_path, batch_size=32)

## 4. Model Definitions

In [ ]:
# ===================== Custom CNN =====================
class CustomCNN(nn.Module):
    def __init__(self, num_classes=9, dropout_rate=0.5):
        super().__init__()
        self.block1 = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2), nn.Dropout2d(0.25))
        self.block2 = nn.Sequential(
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2), nn.Dropout2d(0.25))
        self.block3 = nn.Sequential(
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2), nn.Dropout2d(0.25))
        self.block4 = nn.Sequential(
            nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2), nn.Dropout2d(0.25))
        self.global_pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            nn.Linear(256, 512), nn.ReLU(inplace=True), nn.Dropout(dropout_rate),
            nn.Linear(512, 128), nn.ReLU(inplace=True), nn.Dropout(dropout_rate),
            nn.Linear(128, num_classes))

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.block4(x)
        x = self.global_pool(x)
        x = x.view(x.size(0), -1)
        return self.classifier(x)


# ===================== ResNet50 =====================
class ResNet50Model(nn.Module):
    def __init__(self, num_classes=9, freeze_backbone=True):
        super().__init__()
        self.backbone = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
        if freeze_backbone:
            for name, param in self.backbone.named_parameters():
                if "layer4" not in name and "fc" not in name:
                    param.requires_grad = False
        in_features = self.backbone.fc.in_features
        self.backbone.fc = nn.Sequential(
            nn.Dropout(0.5), nn.Linear(in_features, 512), nn.ReLU(inplace=True),
            nn.Dropout(0.3), nn.Linear(512, num_classes))

    def forward(self, x):
        return self.backbone(x)


# ===================== EfficientNet-B0 =====================
class EfficientNetModel(nn.Module):
    def __init__(self, num_classes=9, freeze_backbone=True):
        super().__init__()
        self.backbone = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
        if freeze_backbone:
            for name, param in self.backbone.named_parameters():
                if "classifier" not in name and "features.8" not in name:
                    param.requires_grad = False
        in_features = self.backbone.classifier[1].in_features
        self.backbone.classifier = nn.Sequential(
            nn.Dropout(0.5), nn.Linear(in_features, 512), nn.ReLU(inplace=True),
            nn.Dropout(0.3), nn.Linear(512, num_classes))

    def forward(self, x):
        return self.backbone(x)


# ===================== ViT-B/16 =====================
class ViTModel(nn.Module):
    def __init__(self, num_classes=9, model_name="vit_base_patch16_224", freeze_backbone=True, dropout_rate=0.3):
        super().__init__()
        self.backbone = timm.create_model(model_name, pretrained=True, num_classes=0)
        if freeze_backbone:
            for name, param in self.backbone.named_parameters():
                if "blocks.11" not in name and "blocks.10" not in name and "norm" not in name:
                    param.requires_grad = False
        feature_dim = self.backbone.num_features
        self.classifier = nn.Sequential(
            nn.LayerNorm(feature_dim), nn.Dropout(dropout_rate),
            nn.Linear(feature_dim, 256), nn.GELU(), nn.Dropout(dropout_rate),
            nn.Linear(256, num_classes))

    def forward(self, x):
        features = self.backbone(x)
        return self.classifier(features)


# Model factory
def get_model(model_name):
    models_map = {
        "custom_cnn": lambda: CustomCNN(num_classes=NUM_CLASSES),
        "resnet50": lambda: ResNet50Model(num_classes=NUM_CLASSES),
        "efficientnet": lambda: EfficientNetModel(num_classes=NUM_CLASSES),
        "vit": lambda: ViTModel(num_classes=NUM_CLASSES),
    }
    return models_map[model_name]()

## 5. Training Functions

In [ ]:
def train_one_epoch(model, train_loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    all_preds, all_labels = [], []

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        all_preds.extend(outputs.argmax(dim=1).cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    return {"loss": running_loss / len(train_loader), "accuracy": accuracy_score(all_labels, all_preds)}


@torch.no_grad()
def validate(model, val_loader, criterion, device):
    model.eval()
    running_loss = 0.0
    all_preds, all_labels = [], []

    for images, labels in val_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)

        running_loss += loss.item()
        all_preds.extend(outputs.argmax(dim=1).cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    return {
        "loss": running_loss / len(val_loader),
        "accuracy": accuracy_score(all_labels, all_preds),
        "precision": precision_score(all_labels, all_preds, average="macro", zero_division=0),
        "recall": recall_score(all_labels, all_preds, average="macro", zero_division=0),
        "f1": f1_score(all_labels, all_preds, average="macro", zero_division=0),
    }

In [ ]:
def train_model(model_name, train_loader, val_loader, class_weights, epochs=30, learning_rate=None, patience=7):
    """Train a single model and save the best checkpoint."""

    # Per-model learning rates
    lr_map = {"custom_cnn": 1e-3, "resnet50": 1e-4, "efficientnet": 1e-4, "vit": 5e-5}
    if learning_rate is None:
        learning_rate = lr_map.get(model_name, 1e-4)

    model = get_model(model_name).to(device)
    criterion = nn.CrossEntropyLoss(weight=class_weights.to(device), label_smoothing=0.1)
    optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                            lr=learning_rate, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2, eta_min=1e-6)

    # Early stopping
    best_val_loss = float("inf")
    es_counter = 0

    os.makedirs("models", exist_ok=True)
    best_val_metrics = None
    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f"\n{'='*60}")
    print(f"Training {model_name} | LR={learning_rate} | Params: {trainable:,}/{total:,} trainable")
    print(f"{'='*60}")

    for epoch in range(1, epochs + 1):
        start = time.time()
        train_metrics = train_one_epoch(model, train_loader, criterion, optimizer, device)
        val_metrics = validate(model, val_loader, criterion, device)
        scheduler.step(epoch)
        elapsed = time.time() - start

        history["train_loss"].append(train_metrics["loss"])
        history["train_acc"].append(train_metrics["accuracy"])
        history["val_loss"].append(val_metrics["loss"])
        history["val_acc"].append(val_metrics["accuracy"])

        improved = ""
        if val_metrics["loss"] < best_val_loss:
            best_val_loss = val_metrics["loss"]
            best_val_metrics = val_metrics
            es_counter = 0
            model_path = f"models/{model_name}_best.pth"
            torch.save({
                "model_state_dict": model.state_dict(),
                "model_name": model_name,
                "epoch": epoch,
                "val_metrics": val_metrics,
                "num_classes": NUM_CLASSES,
            }, model_path)
            improved = " ✓ saved"
        else:
            es_counter += 1

        print(f"Epoch {epoch:2d}/{epochs} ({elapsed:.1f}s) | "
              f"Train Loss: {train_metrics['loss']:.4f} Acc: {train_metrics['accuracy']:.4f} | "
              f"Val Loss: {val_metrics['loss']:.4f} Acc: {val_metrics['accuracy']:.4f} "
              f"F1: {val_metrics['f1']:.4f}{improved}"
              f"{f' (ES {es_counter}/{patience})' if es_counter > 0 else ''}")

        if es_counter >= patience:
            print(f"Early stopping triggered at epoch {epoch}")
            break

    print(f"\nBest {model_name} → Val Acc: {best_val_metrics['accuracy']:.4f}, F1: {best_val_metrics['f1']:.4f}")
    return model, best_val_metrics, history

## 6. Train All Models

Each model trains for 30 epochs. You can adjust `EPOCHS` or skip specific models by commenting them out.

In [ ]:
EPOCHS = 30

# Models to train — comment out any you want to skip
MODELS_TO_TRAIN = [
    "resnet50",
    "efficientnet",
    "vit",
    "custom_cnn",
]

all_results = {}
all_histories = {}

for model_name in MODELS_TO_TRAIN:
    model, metrics, history = train_model(model_name, train_loader, val_loader, class_weights, epochs=EPOCHS)
    all_results[model_name] = metrics
    all_histories[model_name] = history
    # Free GPU memory between models
    del model
    torch.cuda.empty_cache()

## 7. Results Summary

In [ ]:
print(f"\n{'='*70}")
print(f"{'Model':<15} {'Accuracy':>10} {'Precision':>10} {'Recall':>10} {'F1':>10}")
print(f"{'-'*70}")
for name, m in all_results.items():
    print(f"{name:<15} {m['accuracy']:>10.4f} {m['precision']:>10.4f} {m['recall']:>10.4f} {m['f1']:>10.4f}")
print(f"{'='*70}")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for name, h in all_histories.items():
    axes[0].plot(h["train_loss"], label=f"{name} train")
    axes[0].plot(h["val_loss"], "--", label=f"{name} val")
axes[0].set_title("Loss")
axes[0].set_xlabel("Epoch")
axes[0].legend()
axes[0].grid(True)

for name, h in all_histories.items():
    axes[1].plot(h["val_acc"], label=name)
axes[1].set_title("Validation Accuracy")
axes[1].set_xlabel("Epoch")
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.savefig("training_curves.png", dpi=150)
plt.show()

## 8. Evaluate on Test Set

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

def evaluate_on_test(model_name, test_loader):
    model = get_model(model_name).to(device)
    ckpt = torch.load(f"models/{model_name}_best.pth", map_location=device)
    model.load_state_dict(ckpt["model_state_dict"])
    model.eval()

    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            outputs = model(images)
            all_preds.extend(outputs.argmax(dim=1).cpu().numpy())
            all_labels.extend(labels.numpy())

    print(f"\n{'='*60}")
    print(f"Test Results — {model_name}")
    print(f"{'='*60}")
    print(classification_report(all_labels, all_preds, target_names=CLASS_NAMES, zero_division=0))

    # Confusion matrix
    cm = confusion_matrix(all_labels, all_preds)
    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
    ax.set_title(f"Confusion Matrix — {model_name}")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()

    return all_preds, all_labels


for model_name in MODELS_TO_TRAIN:
    evaluate_on_test(model_name, test_loader)

## 9. Download Trained Models

Run this cell to download the `.pth` files to your local machine.

In [ ]:
from google.colab import files

for model_name in MODELS_TO_TRAIN:
    path = f"models/{model_name}_best.pth"
    if os.path.exists(path):
        print(f"Downloading {path}...")
        files.download(path)
    else:
        print(f"WARNING: {path} not found")